# GRB preprocessing report

Inspect the cached Swift GRB light curves, compare short and long GRBs, and print one full example from each class.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from classes import GRBHDF5Dataset

PROJECT_ROOT = Path.cwd()
H5_FILE = PROJECT_ROOT / "data" / "raw" / "classipygrb" / "swift_balanced_lightcurves.h5"
CLASS_NAMES = {0: "short", 1: "long"}

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

In [2]:
def class_label(label: int) -> str:
    return CLASS_NAMES.get(int(label), f"class_{label}")


def as_numpy(dataset: GRBHDF5Dataset) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = dataset.x.detach().cpu().numpy()
    labels = np.asarray(dataset.labels, dtype=int)
    t90 = np.asarray(dataset.t90, dtype=float)
    return x, labels, t90


def class_distribution(labels: np.ndarray) -> pd.DataFrame:
    rows = []
    total = labels.size
    for label in sorted(np.unique(labels)):
        count = int((labels == label).sum())
        rows.append(
            {
                "label": int(label),
                "class": class_label(int(label)),
                "count": count,
                "percent": 100.0 * count / total if total else 0.0,
            }
        )
    return pd.DataFrame(rows)


def describe_values(values: np.ndarray) -> dict[str, float]:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "median": np.nan,
            "min": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "max": np.nan,
        }

    return {
        "mean": float(np.mean(finite)),
        "std": float(np.std(finite)),
        "median": float(np.median(finite)),
        "min": float(np.min(finite)),
        "q25": float(np.percentile(finite, 25)),
        "q75": float(np.percentile(finite, 75)),
        "max": float(np.max(finite)),
    }


def t90_summary(labels: np.ndarray, t90: np.ndarray) -> pd.DataFrame:
    rows = []
    for label in sorted(np.unique(labels)):
        stats = describe_values(t90[labels == label])
        rows.append({"class": class_label(int(label)), **stats})
    return pd.DataFrame(rows)

In [13]:
def overall_signal_summary(x: np.ndarray, labels: np.ndarray) -> pd.DataFrame:
    rows = []
    for label in sorted(np.unique(labels)):
        class_x = x[labels == label]
        rows.append(
            {
                "class": class_label(int(label)),
                "all_values_mean": float(np.mean(class_x)),
                "all_values_std": float(np.std(class_x)),
                "per_grb_mean": float(np.mean(class_x.mean(axis=(1, 2)))),
                "per_grb_peak_mean": float(np.mean(class_x.max(axis=(1, 2)))),
                "per_grb_sum_mean": float(np.mean(class_x.sum(axis=(1, 2)))),
                "zero_fraction": float(np.mean(class_x == 0.0)),
            }
        )
    return pd.DataFrame(rows)


def band_summary(x: np.ndarray, labels: np.ndarray, channel_columns: list[str]) -> pd.DataFrame:
    rows = []
    for label in sorted(np.unique(labels)):
        class_x = x[labels == label]
        for channel_idx, channel_name in enumerate(channel_columns):
            channel_x = class_x[:, :, channel_idx]
            per_grb_mean = channel_x.mean(axis=1)
            per_grb_peak = channel_x.max(axis=1)
            per_grb_sum = channel_x.sum(axis=1)
            rows.append(
                {
                    "class": class_label(int(label)),
                    "band": channel_name,
                    "timepoint_mean": float(np.mean(channel_x)),
                    "timepoint_std": float(np.std(channel_x)),
                    "per_grb_mean_avg": float(np.mean(per_grb_mean)),
                    "per_grb_peak_avg": float(np.mean(per_grb_peak)),
                    "per_grb_sum_avg": float(np.mean(per_grb_sum)),
                    "zero_fraction": float(np.mean(channel_x == 0.0)),
                }
            )
    return pd.DataFrame(rows)


def short_long_difference(bands: pd.DataFrame) -> pd.DataFrame:
    if not {"short", "long"}.issubset(set(bands["class"])):
        return pd.DataFrame()

    numeric_columns = [
        "timepoint_mean",
        "timepoint_std",
        "per_grb_mean_avg",
        "per_grb_peak_avg",
        "per_grb_sum_avg",
        "zero_fraction",
    ]
    short = bands[bands["class"] == "short"].set_index("band")
    long = bands[bands["class"] == "long"].set_index("band")
    rows = []
    for band in short.index.intersection(long.index):
        row = {"band": band}
        for column in numeric_columns:
            short_value = float(short.loc[band, column])
            long_value = float(long.loc[band, column])
            row[f"{column}_long_minus_short"] = long_value - short_value
            row[f"{column}_long_div_short"] = (
                long_value / short_value if not np.isclose(short_value, 0.0) else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows)

In [4]:
def full_grb_dataframe(
    dataset: GRBHDF5Dataset,
    x: np.ndarray,
    labels: np.ndarray,
    label: int,
) -> tuple[str, float, pd.DataFrame]:
    indices = np.where(labels == label)[0]
    if indices.size == 0:
        raise ValueError(f"No {class_label(label)} GRB found")

    idx = int(indices[0])
    grb = pd.DataFrame(x[idx], columns=dataset.channel_columns)
    grb.insert(0, "time_index", np.arange(len(grb)))
    return dataset.names[idx], dataset.t90[idx], grb

## Load dataset

In [5]:
dataset = GRBHDF5Dataset(H5_FILE)
x, labels, t90 = as_numpy(dataset)

print(f"HDF5 file: {H5_FILE}")
print(f"Loaded GRBs: {len(dataset)}")
print(f"Input shape: {tuple(x.shape)}")
print(f"Channels: {', '.join(dataset.channel_columns)}")
print(f"Label rule: {dataset.label_rule}")
print(f"Cache normalization: {dataset.cache_normalization}")

HDF5 file: /Users/daniele/Desktop/esami mancanti/grb-thesis/Spectral-Evolution-of-a-GRB/data/raw/classipygrb/swift_balanced_lightcurves.h5
Loaded GRBs: 268
Input shape: (268, 256, 4)
Channels: 15-25keV, 25-50keV, 50-100keV, 100-350keV
Label rule: 0 short: T90 <= 2s; 1 long: T90 > 2s
Cache normalization: none; raw resampled light curves


## Class distribution

In [6]:
class_distribution(labels)

,label,class,count,percent
0,0,short,134,50.0000
1,1,long,134,50.0000


## How time, channels, and T90 fit together

- `t90` is one number for the whole GRB. It is the duration used to label the burst as short or long.
- `time_index` is only the row position inside the processed light curve array. It is not T90 and it is not necessarily seconds.
- Each channel/band column is one energy band value at that processed time position.
- The HDF5 file stores a fixed-size tensor, so every GRB has the same number of time rows. Short bursts often have many zero rows because the useful signal is short compared with the fixed length.

In [ ]:
print(f"X shape: {x.shape}")
print("Meaning: (number_of_grbs, processed_time_rows, channels)")
print(f"Number of GRBs: {x.shape[0]}")
print(f"Processed time rows per GRB: {x.shape[1]}")
print(f"Number of channels/bands: {x.shape[2]}")
print(f"Channels: {dataset.channel_columns}")

pd.DataFrame(
    {
        "object": ["t90", "time_index", "channel value"],
        "where_it_lives": ["dataset.t90, one value per GRB", "row index inside X", "X[grb_index, time_index, channel_index]"],
        "meaning": [
            "duration of the burst; used for short/long label",
            "processed row number; not automatically seconds",
            "light-curve value for one band at one processed time row",
        ],
    }
)

In [ ]:
def nonzero_row_summary(grb_df: pd.DataFrame) -> dict[str, int | None]:
    band_cols = [column for column in grb_df.columns if column != "time_index"]
    nonzero_rows = (grb_df[band_cols] != 0).any(axis=1)

    if not nonzero_rows.any():
        first_nonzero = None
        last_nonzero = None
    else:
        nonzero_indices = grb_df.loc[nonzero_rows, "time_index"]
        first_nonzero = int(nonzero_indices.iloc[0])
        last_nonzero = int(nonzero_indices.iloc[-1])

    return {
        "total_rows": int(len(grb_df)),
        "rows_with_any_signal": int(nonzero_rows.sum()),
        "all_zero_rows": int((~nonzero_rows).sum()),
        "first_nonzero_time_index": first_nonzero,
        "last_nonzero_time_index": last_nonzero,
    }


short_name, short_t90, short_grb = full_grb_dataframe(dataset, x, labels, label=0)
long_name, long_t90, long_grb = full_grb_dataframe(dataset, x, labels, label=1)

pd.DataFrame(
    [
        {"class": "short", "name": short_name, "t90": short_t90, **nonzero_row_summary(short_grb)},
        {"class": "long", "name": long_name, "t90": long_t90, **nonzero_row_summary(long_grb)},
    ]
)

In [ ]:
def compact_grb_preview(grb_df: pd.DataFrame, edge_rows: int = 8) -> pd.DataFrame:
    if len(grb_df) <= edge_rows * 2:
        return grb_df

    separator = pd.DataFrame(
        [{column: "..." for column in grb_df.columns}],
        columns=grb_df.columns,
    )
    return pd.concat([grb_df.head(edge_rows), separator, grb_df.tail(edge_rows)], ignore_index=True)


print(f"Short GRB preview: {short_name}, t90={short_t90:.4g}")
display(compact_grb_preview(short_grb))

print(f"Long GRB preview: {long_name}, t90={long_t90:.4g}")
display(compact_grb_preview(long_grb))

## Are the zeros saved in HDF5 or created by ClassiPy?

The zeros you see here are already inside the HDF5 `X` dataset loaded from disk. They are not created by the notebook display.

ClassiPyGRB provides the raw Swift light-curve data. In this project, before saving the HDF5 cache, the light curves were converted to fixed-length arrays. During that preprocessing, short or incomplete curves can be padded with zeros, and missing/non-finite values can also become zeros. So the zeros are saved in HDF5, but they mainly come from the project preprocessing step rather than being a new effect of this notebook.

In [ ]:
zero_report = []
for label in sorted(np.unique(labels)):
    class_x = x[labels == label]
    zero_report.append(
        {
            "class": class_label(int(label)),
            "values_in_h5_X": int(class_x.size),
            "zeros_in_h5_X": int((class_x == 0).sum()),
            "zero_fraction": float((class_x == 0).mean()),
        }
    )

pd.DataFrame(zero_report)

## Full short GRB example

In [7]:
short_name, short_t90, short_grb = full_grb_dataframe(dataset, x, labels, label=0)
print(f"Short GRB: name={short_name}, label=0, t90={short_t90:.4g}")

with pd.option_context("display.max_rows", None):
    display(short_grb)

Short GRB: name=GRB101219A, label=0, t90=0.828


,time_index,15-25keV,25-50keV,50-100keV,100-350keV
0,0,-0.0768,-0.1466,0.0023,-0.0023
1,1,-0.0130,0.0784,0.0265,0.0234
2,2,0.0521,0.0378,-0.0074,0.0185
3,3,-0.0158,-0.0412,0.0420,0.0641
4,4,-0.0638,-0.0031,0.0179,-0.0230
5,5,0.0091,0.0134,-0.0012,-0.0544
6,6,0.0419,-0.0084,0.0181,-0.0301
7,7,0.0279,0.0072,0.0048,-0.0654
8,8,0.0540,-0.0751,0.0516,0.0146
9,9,0.0201,0.0696,0.0811,-0.0015


## Full long GRB example

In [8]:
long_name, long_t90, long_grb = full_grb_dataframe(dataset, x, labels, label=1)
print(f"Long GRB: name={long_name}, label=1, t90={long_t90:.4g}")

with pd.option_context("display.max_rows", None):
    display(long_grb)

Long GRB: name=GRB180626A, label=1, t90=30.1


,time_index,15-25keV,25-50keV,50-100keV,100-350keV
0,0,0.0054,-0.0199,0.0126,-0.0294
1,1,-0.0271,-0.0021,0.0256,0.0286
2,2,0.0049,-0.0508,-0.0290,-0.0013
3,3,-0.0053,0.0028,-0.0198,0.0138
4,4,-0.0221,-0.0008,0.0171,-0.0373
5,5,-0.0505,0.0266,0.0069,0.0017
6,6,0.0448,-0.0395,-0.0390,0.0384
7,7,0.0304,-0.0385,-0.0069,0.0096
8,8,-0.0002,0.0363,0.0046,0.0199
9,9,0.0237,-0.0328,-0.0236,0.0108


## T90 summary by class

In [9]:
t90_summary(labels, t90)

,class,mean,std,median,min,q25,q75,max
0,short,0.5541,0.4839,0.3840,0.0120,0.1800,0.8250,2.0000
1,long,79.8882,94.2413,44.7980,2.0920,17.8240,99.1700,600.0080


## Overall light-curve summary by class

In [10]:
overall_signal_summary(x, labels)

,class,all_values_mean,all_values_std,per_grb_mean,per_grb_peak_mean,per_grb_sum_mean,zero_fraction
0,short,0.0005,0.0227,0.0005,0.1156,0.5148,0.1200
1,long,0.0021,0.0290,0.0021,0.1681,2.1908,0.1467


## Average values across bands by class

In [11]:
bands = band_summary(x, labels, dataset.channel_columns)
bands

,class,band,timepoint_mean,timepoint_std,per_grb_mean_avg,per_grb_peak_avg,per_grb_sum_avg,zero_fraction
0,short,15-25keV,0.0001,0.0233,0.0001,0.0806,0.0131,0.1200
1,short,25-50keV,0.0008,0.0249,0.0008,0.0969,0.2005,0.1200
2,short,50-100keV,0.0009,0.0229,0.0009,0.0915,0.2184,0.1200
3,short,100-350keV,0.0003,0.0193,0.0003,0.0668,0.0827,0.1200
4,long,15-25keV,0.0022,0.0283,0.0022,0.1175,0.5539,0.1467
5,long,25-50keV,0.0032,0.0337,0.0032,0.1503,0.8112,0.1467
6,long,50-100keV,0.0027,0.0312,0.0027,0.1354,0.7015,0.1467
7,long,100-350keV,0.0005,0.0210,0.0005,0.0808,0.1241,0.1467


## Long versus short differences by band

In [12]:
short_long_difference(bands)

,band,timepoint_mean_long_minus_short,timepoint_mean_long_div_short,timepoint_std_long_minus_short,timepoint_std_long_div_short,per_grb_mean_avg_long_minus_short,per_grb_mean_avg_long_div_short,per_grb_peak_avg_long_minus_short,per_grb_peak_avg_long_div_short,per_grb_sum_avg_long_minus_short,per_grb_sum_avg_long_div_short,zero_fraction_long_minus_short,zero_fraction_long_div_short
0,15-25keV,0.0021,42.3769,0.0050,1.2166,0.0021,42.3769,0.0369,1.4583,0.5409,42.3769,0.0267,1.2225
1,25-50keV,0.0024,4.0453,0.0088,1.3543,0.0024,4.0453,0.0534,1.5508,0.6107,4.0453,0.0267,1.2225
2,50-100keV,0.0019,3.2115,0.0083,1.3616,0.0019,3.2115,0.0439,1.4803,0.4831,3.2115,0.0267,1.2225
3,100-350keV,0.0002,1.4993,0.0017,1.0891,0.0002,1.4993,0.0141,1.2110,0.0413,1.4993,0.0267,1.2222
